# E-Commerce Sales & Customer Analytics
## Exploratory Data Analysis (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 1. Load Data

In [ ]:
df = pd.read_csv('data/processed/ecommerce_processed.csv', parse_dates=['Order Date'])
print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

## 2. Data Overview

In [ ]:
print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"\nDate Range: {df['Order Date'].min()} to {df['Order Date'].max()}")
print(f"Total Orders: {df['Order ID'].nunique()}")
print(f"Unique Customers: {df['Customer ID'].nunique()}")
print(f"Unique Products: {df['Product Name'].nunique()}")
print(f"Unique Categories: {df['Category'].nunique()}")
print(f"Unique Regions: {df['Region'].nunique()}")

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

## 3. Key Metrics

In [ ]:
total_revenue = df['Sales'].sum()
total_profit = df['Profit'].sum()
total_orders = len(df)
profit_margin = (total_profit / total_revenue) * 100

print("=" * 50)
print("KEY BUSINESS METRICS")
print("=" * 50)
print(f"Total Revenue:     ${total_revenue:,.2f}")
print(f"Total Profit:      ${total_profit:,.2f}")
print(f"Total Orders:      {total_orders:,}")
print(f"Profit Margin:     {profit_margin:.2f}%")
print(f"Avg Order Value:   ${total_revenue/total_orders:.2f}")

## 4. Monthly Sales Trend

In [ ]:
monthly = df.groupby(['Year', 'Month']).agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order ID': 'count'
}).reset_index()
monthly['Year-Month'] = monthly['Year'].astype(str) + '-' + monthly['Month'].astype(str).str.zfill(2)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(monthly['Year-Month'], monthly['Sales'], marker='o', linewidth=2, color='#2E86AB')
axes[0].set_title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Revenue ($)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].fill_between(monthly['Year-Month'], monthly['Sales'], alpha=0.3, color='#2E86AB')

axes[1].bar(monthly['Year-Month'], monthly['Profit'], color='#28A745', alpha=0.8)
axes[1].set_title('Monthly Profit Trend', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Profit ($)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('reports/monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Sales by Category

In [ ]:
category_sales = df.groupby('Category').agg({
    'Sales': 'sum',
    'Profit': 'sum'
}).sort_values('Sales', ascending=False)
category_sales['Margin'] = (category_sales['Profit'] / category_sales['Sales'] * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = sns.color_palette('viridis', len(category_sales))
axes[0].pie(category_sales['Sales'], labels=category_sales.index, autopct='%1.1f%%', 
            colors=colors, startangle=90)
axes[0].set_title('Revenue by Category', fontsize=14, fontweight='bold')

bars = axes[1].barh(category_sales.index, category_sales['Sales'], color=colors)
axes[1].set_title('Revenue by Category', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Revenue ($)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('reports/category_sales.png', dpi=150, bbox_inches='tight')
plt.show()

print(category_sales)

## 6. Sales by Region

In [ ]:
region_sales = df.groupby('Region').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order ID': 'count'
}).sort_values('Sales', ascending=False)
region_sales['Margin'] = (region_sales['Profit'] / region_sales['Sales'] * 100).round(2)

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(region_sales))
width = 0.35

bars1 = ax.bar(x - width/2, region_sales['Sales'], width, label='Revenue', color='#2E86AB')
bars2 = ax.bar(x + width/2, region_sales['Profit'], width, label='Profit', color='#28A745')

ax.set_xlabel('Region', fontsize=12)
ax.set_ylabel('Amount ($)', fontsize=12)
ax.set_title('Revenue & Profit by Region', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(region_sales.index)
ax.legend()

plt.tight_layout()
plt.savefig('reports/region_sales.png', dpi=150, bbox_inches='tight')
plt.show()

print(region_sales)

## 7. Top 10 Products

In [ ]:
top_products = df.groupby('Product Name').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Quantity': 'sum'
}).sort_values('Sales', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.barh(top_products.index, top_products['Sales'], color=sns.color_palette('magma', 10))
ax.set_xlabel('Revenue ($)', fontsize=12)
ax.set_title('Top 10 Products by Revenue', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for bar, val in zip(bars, top_products['Sales']):
    ax.text(val + 10000, bar.get_y() + bar.get_height()/2, f'${val:,.0f}', 
            va='center', fontsize=10)

plt.tight_layout()
plt.savefig('reports/top_products.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Top 10 Customers

In [ ]:
top_customers = df.groupby(['Customer ID', 'Customer Name']).agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order ID': 'count'
}).sort_values('Sales', ascending=False).head(10)
top_customers.columns = ['Revenue', 'Profit', 'Orders']

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.barh(top_customers['Customer Name'], top_customers['Revenue'], color=sns.color_palette('crest', 10))
ax.set_xlabel('Total Revenue ($)', fontsize=12)
ax.set_title('Top 10 Customers by Revenue', fontsize=14, fontweight='bold')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('reports/top_customers.png', dpi=150, bbox_inches='tight')
plt.show()

print(top_customers)

## 9. Discount Impact Analysis

In [ ]:
discount_analysis = df.groupby('Discount').agg({
    'Sales': ['sum', 'mean'],
    'Profit': ['sum', 'mean'],
    'Order ID': 'count'
}).round(2)
discount_analysis.columns = ['Total Sales', 'Avg Sales', 'Total Profit', 'Avg Profit', 'Orders']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['Discount'], df['Sales'], alpha=0.3, color='#2E86AB')
axes[0].set_xlabel('Discount (%)')
axes[0].set_ylabel('Sales ($)')
axes[0].set_title('Discount vs Sales', fontsize=14, fontweight='bold')

discount_groups = df.groupby(pd.cut(df['Discount'], bins=[0, 10, 20, 30, 100])).agg({'Profit': 'mean'}).dropna()
axes[1].bar(discount_groups.index.astype(str), discount_groups['Profit'], color=['#28A745', '#FFC107', '#DC3545', '#6C757D'])
axes[1].set_xlabel('Discount Range (%)')
axes[1].set_ylabel('Average Profit ($)')
axes[1].set_title('Avg Profit by Discount Range', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('reports/discount_impact.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Correlation Analysis

In [ ]:
numeric_cols = ['Quantity', 'Sales', 'Profit', 'Discount', 'Profit Margin', 'Unit Price']
correlation = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, fmt='.2f', 
            square=True, linewidths=0.5)
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('reports/correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Key Insights Summary

In [ ]:
print("=" * 60)
print("KEY BUSINESS INSIGHTS")
print("=" * 60)

print("\n📊 REVENUE OVERVIEW:")
print(f"   - Total Revenue: ${total_revenue:,.2f}")
print(f"   - Total Profit: ${total_profit:,.2f}")
print(f"   - Profit Margin: {profit_margin:.2f}%")

print("\n🏆 TOP CATEGORY:")
top_cat = category_sales['Sales'].idxmax()
print(f"   - {top_cat}: ${category_sales.loc[top_cat, 'Sales']:,.2f}")

print("\n🌍 TOP REGION:")
top_reg = region_sales['Sales'].idxmax()
print(f"   - {top_reg}: ${region_sales.loc[top_reg, 'Sales']:,.2f}")

print("\n📦 TOP PRODUCT:")
top_prod = top_products['Sales'].idxmax()
print(f"   - {top_prod}: ${top_products.loc[top_prod, 'Sales']:,.2f}")

print("\n💰 TOP CUSTOMER:")
top_cust_name = top_customers.index[0][1]
top_cust_rev = top_customers.iloc[0]['Revenue']
print(f"   - {top_cust_name}: ${top_cust_rev:,.2f}")

print("\n📉 DISCOUNT INSIGHT:")
high_disc = df[df['Discount'] >= 20]['Profit Margin'].mean()
low_disc = df[df['Discount'] < 10]['Profit Margin'].mean()
print(f"   - Orders with <10% discount avg margin: {low_disc:.2f}%")
print(f"   - Orders with >=20% discount avg margin: {high_disc:.2f}%")